In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.signal
import matplotlib.pyplot as plt

In [ ]:
def ECSA_calc(loc,fnam):
    ECSA = 0
    files = os.listdir(loc)
    elem = pd.DataFrame
    liz = [elem,elem,elem,elem,elem,elem]
    counter = 2
    for i in range(len(files)):
        if ((("0"+str(counter)+'_CV') in files[i]) and ('txt' in files[i])):
            print(files[i])
            #saving dataframe
            liz[counter-2] = pd.read_csv(loc+'/'+files[i],encoding="latin1", sep="\s+", decimal=",")
            #determining amount of cycles
            last_cycle = np.max(liz[counter-2]['cycle'])
            #selecting last cycle for data
            liz[counter-2] = liz[counter-2].drop(liz[counter-2][liz[counter-2]['cycle'] < last_cycle].index)
            counter += 1
    if counter == 8:
        scan_rate = [20, 50, 100, 150, 200, 250]
        i_avg = np.zeros(len(liz))
        colors = ['red', 'blue', 'black', 'magenta', 'orange','green','lime']
        for G in range(len(liz)):
            x = liz[G]['Ewe/V']
            y = liz[G]['<I>/mA']
            #Cutting off the segment that is above 0 mA
            y_pos_idx = y[y > 0]
            #extracting all the index values
            idx_pos = y_pos_idx.index
            #generating the initial arrays
            x_pos = np.zeros(len(idx_pos))
            y_pos = np.zeros(len(idx_pos))
            #running for loop to allocate x and y values for all the y values that are below 0, and storing them in an array for later use
            for i in range(len(idx_pos)):
                x_pos[i] = x[idx_pos[i]]
                y_pos[i] = y[idx_pos[i]]
            #determining index of the lowest absolute value of x
            dx_pos = np.absolute(x_pos).argmin()
            #Cutting off the segment that is below 0 mA
            y_neg_idx = y[y < 0]
            #extracting all the index values
            idx_neg = y_neg_idx.index
            #generating the initial arrays
            x_neg = np.zeros(len(idx_neg))
            y_neg = np.zeros(len(idx_neg))
            #running for loop to allocate x and y values for all the y values that are below 0, and storing them in an array for later use
            for i in range(len(idx_neg)):
                x_neg[i] = x[idx_neg[i]]
                y_neg[i] = y[idx_neg[i]]
            #deteriming the index of the lowest absolute value of x
            dx_neg = np.absolute(x_neg).argmin() 
            
            #determining the average of the current density at potential = approx. 0 
            i_avg[G] = np.mean(np.absolute([y_neg[dx_neg], y_pos[dx_pos]]))
            
            plt.plot(x,y,color=colors[G],label=scan_rate[G])
            plt.plot(x_neg[dx_neg],y_neg[dx_neg],'x',color=colors[G])
            plt.plot(x_pos[dx_pos],y_pos[dx_pos],'x',color=colors[G])    
        plt.legend()
        plt.grid()
        plt.xlabel('[Ewe/V]')
        plt.ylabel('[<I>/mA]')
        plt.title("Cyclic voltametry at specific refresh rates "+fnam)

        plt.savefig('./results/CV_'+fnam+'.png')
        plt.show()
        
        x = liz[1]['Ewe/V']
        y = liz[1]['<I>/mA']
        plt.plot(x,y,color=colors[1],label=str(scan_rate[1])+" mV/s")
        x = liz[3]['Ewe/V']
        y = liz[3]['<I>/mA']
        plt.plot(x,y,color=colors[2],label=str(scan_rate[3])+" mV/s")
        x = liz[-1]['Ewe/V']
        y = liz[-1]['<I>/mA']
        plt.plot(x,y,color=colors[0],label=str(scan_rate[-1])+" mV/s")
        plt.title("Example capactiance measurement")
        plt.xlabel('Applied potential (V)')
        plt.ylabel('Measured current (mA)')
        plt.legend()
        plt.grid()
        plt.show()
        #determining slope --> EDLC
        plt.plot(scan_rate,i_avg,'x', label="experimental data")
        pars = np.polyfit(scan_rate,i_avg,1)
        z = np.polyval(pars,[0,250])
        EDLC = pars[0]
        plt.plot([0,250],z, label="polyfit 1st deg")
        plt.xlabel('refresh rate in [mV/s]')
        plt.ylabel("Average current density in [mA]")
        plt.title("Determining EDLC: "+str(EDLC)[:10]+" Farad")
        plt.legend()
        plt.savefig('./results/EDLC_'+fnam+'.png')
        plt.show()
        
        #determining ECSA from EDLC
        Cspec = 23*10**-6 #micro F cm^-2
        ECSA = EDLC/Cspec
        ECSA = ECSA/10000
    
    return(ECSA)

In [ ]:
def first_PEIS(loc,fnam):
    files = os.listdir(loc)
    for i in range(len(files)):
        if (('01_PEIS') in files[i] and ('txt' in files[i])):
            print(files[i])
            PEIS_01 = pd.read_csv(loc+'/'+files[i],encoding="latin1", sep="\s+", decimal=",")
            #Plotting Initial PEIS scan
            plt.scatter(x=PEIS_01['Re(Z)/Ohm'], y=PEIS_01['-Im(Z)/Ohm'],c=PEIS_01['freq/Hz'],marker='s', cmap='RdBu' )
            plt.axis('equal')
            plt.grid()
            plt.title("EIS - Initial scan "+fnam)
            plt.xlabel("Re(Z)/Ohm")
            plt.ylabel("-Im(Z)/Ohm")
            plt.colorbar(label="frequency")
            plt.savefig('./results/PEIS1_'+fnam+'.png')
            plt.show()
    return('Analysis complete')

In [ ]:
def second_PEIS(loc,fnam):
    mass_resistance = 0
    electrode_res = 0
    electrolyte_res = 0
    files = os.listdir(loc)
    R_PEIS= 0
    R_Electrode = 0
    R_Electrolyte = 0 
    R_Mass_transfer = 0
    
    for i in range(len(files)):
        if ((('PEIS') in files[i]) and ('01_PEIS' not in files[i]) and ('txt' in files[i])):
            print(files[i])
            PEIS_02 = pd.read_csv(loc+'/'+files[i],encoding="latin1", sep="\s+", decimal=",")
            PEIS_02 = PEIS_02.drop(PEIS_02[PEIS_02["-Im(Z)/Ohm"] < 0].index)
            plt.scatter(x=PEIS_02['Re(Z)/Ohm'], y=PEIS_02['-Im(Z)/Ohm'],c=PEIS_02['freq/Hz'],marker='x', cmap='RdBu' )
            #Creating a rough fit to the data to determine minima
            pars = np.polyfit(PEIS_02['Re(Z)/Ohm'],PEIS_02['-Im(Z)/Ohm'],25)
            y = np.polyval(pars,PEIS_02['Re(Z)/Ohm'])
            #Determining minima --> determining resistances
            #determining electrode resistance ergo the lowest point
            electrode_res = np.min(PEIS_02['Re(Z)/Ohm'])
            plt.axvline(electrode_res,color="red",label='Ohmic resistance')
            min2 = scipy.signal.argrelmin(y, axis=0, order=3, mode='clip')
            if len(min2[0]) <= 0:
                print('electrode & mass transfer resistances cannot be determined, no minima detected')
            else:
                electrolyte_res = float(PEIS_02['Re(Z)/Ohm'].iloc[[min2[0][0]]])
                #Load into new dataframe, the second curve
                mass_res_df = PEIS_02.drop(PEIS_02[ PEIS_02['Re(Z)/Ohm'] < float(PEIS_02['Re(Z)/Ohm'].iloc[[min2[0][0]]])].index)
                #resetting the index
                mass_res_df = mass_res_df.reset_index(drop=True)
                #Dropping all data in the dataframe, that occurrs before the maximum is reached
                mass_res_df = mass_res_df.drop(mass_res_df[ mass_res_df['Re(Z)/Ohm'] < float(mass_res_df['Re(Z)/Ohm'].iloc[[np.argmax(mass_res_df["-Im(Z)/Ohm"])]])].index)
                #determining the value of the new minimum
                mass_res = mass_res_df['Re(Z)/Ohm'].iloc[[np.argmin(mass_res_df['-Im(Z)/Ohm'])]]
                #saving the value as a float
                mass_resistance = float(mass_res)
                mass_idx = PEIS_02[PEIS_02['Re(Z)/Ohm'].values == mass_resistance].index[0]
                electrolyte_idx = PEIS_02[PEIS_02['Re(Z)/Ohm'].values == electrolyte_res].index[0]
                if mass_idx <= electrolyte_idx:
                    print('ups - something went wrong with the experiment! electrolyte & mass resistances cannot be calculated')
                    mass_resistance = 0
                    electrolyte_res = 0
                else:
                    plt.axvline(electrolyte_res,color="blue",label='kinetic resistance')
                    plt.axvline(mass_resistance,color="black",label='mass transfer resistance')
           

            #plotting the nyquist plot
            plt.axis("equal")
            plt.grid()
            plt.legend( loc = 'lower right')
            plt.xlabel("Re(Z)/Ohm")
            plt.ylabel("-Im(Z)/Ohm")
            plt.title("Nyquist plot for "+fnam)
            plt.colorbar(label="frequency")
            #plt.figure(figsize=(50,50))
            plt.savefig('./results/imp_'+fnam+'.png')
            plt.show()
            R_PEIS= mass_resistance
            R_Electrode = electrode_res
            if electrolyte_res > 0:
                R_Electrolyte = electrolyte_res- electrode_res
            else:
                R_Electrolyte = 0 
            R_Mass_transfer = mass_resistance - electrolyte_res

    return(R_PEIS,R_Electrode,R_Electrolyte,R_Mass_transfer)

In [ ]:
def PCGA(loc,fnam):
    PCGA_tot = 0
    cur_good = 0
    pot_good = 0
    PCGA_df = 0
    files = os.listdir(loc)
    for i in range(len(files)):
        if ((("PCGA" in files[i]) or (('arization' in files[i]) and ('imp' not in files[i])) or (('pol' in files[i]) and ('imp' not in files[i]))) and ('txt' in files[i])):
            print(files[i])
            PCGA = pd.read_csv(loc+'/'+files[i],encoding="latin1", sep="\s+", decimal=",")
            #Plotting & Determining total resistance based on PCGA
            #determining when the jumps occur
            jumps=[i for i in range(len(PCGA['Ewe/V'])-1) if abs(PCGA['Ewe/V'][i]-PCGA['Ewe/V'][i+1])>0.01]
            #based on the tollerance at which a jump is measured, values can occur in between jumps, these are now removed
            def isonebigger(a,b):
                return True if abs(a-b)==1 else False
            count=0
            jumps_new=[]
            last_num=-10000
            while True:
                if len(jumps_new)==0:
                    jumps_new.append(jumps[count])
                    count+=1
                else:
                    try:
                        if isonebigger(jumps_new[-1],jumps[count]) or isonebigger(last_num,jumps[count]):
                            last_num=jumps[count]
                            count+=1
                        else:
                            jumps_new.append(jumps[count])
                            count+=1
                    except IndexError:
                        break

            jumps = jumps_new

            #preallocating variables
            slope=[]
            pot=[]

            #determining mean of the second part (ergo final) of each individual jump
            for i in range(len(jumps)):
                if i ==0:
                    cur_data=PCGA['<I>/mA'].values[int(jumps[i]/2):jumps[i]]
                    pot_data=PCGA['Ewe/V'].values[int(jumps[i]/2):jumps[i]]
                else:
                    cur_data=PCGA['<I>/mA'].values[int(jumps[i-1]+(jumps[i]-jumps[i-1])/2):jumps[i]]
                    pot_data=PCGA['Ewe/V'].values[int(jumps[i-1]+(jumps[i]-jumps[i-1])/2):jumps[i]]
                slope.append(np.mean(cur_data))
                pot.append(np.mean(pot_data))
            #determining mean of the final jump
            cur_data=PCGA['<I>/mA'].values[int(jumps[-1]+(len(PCGA['<I>/mA'])-jumps[-1])/2):-1]
            pot_data=PCGA['Ewe/V'].values[int(jumps[-1]+(len(PCGA['Ewe/V'])-jumps[-1])/2):-1]
            slope.append(np.mean(cur_data))
            pot.append(np.mean(pot_data))

            #performing a linear fit on the slope data, as V = IR, and slope is dy/dx; so if y=V and x=I; R = dV/dI
            pars = np.polyfit(slope,pot,1)

            PCGA_tot = pars[0]*1000 #pars[0] * 1000, as I is in milli Ampere
            #generating x and y values for the sloping stuff, this is just to plot this data
            y = np.polyval(pars,[np.min(PCGA['<I>/mA']),np.max(PCGA['<I>/mA'])])
            x = np.linspace(np.min(PCGA['<I>/mA']),np.max(PCGA['<I>/mA']), len(y))

            #plotting; polarization curve
            plt.title('Polarization curve, slope: '+ str(pars[0]*1000)[:10]+" [Ohm]")
            plt.plot(PCGA['<I>/mA'],PCGA['Ewe/V'],'x',label="experimental data")
            #plotting jump "start" crosses
            jumps.append(len(PCGA['<I>/mA'])-1)
            cur_good = np.zeros(len(jumps))
            pot_good = np.zeros(len(jumps))
            for i in range(len(jumps)):
                plt.plot(PCGA['<I>/mA'][jumps[i]],PCGA['Ewe/V'][[jumps[i]]],'x',color="red")
                cur_good[i] = PCGA['<I>/mA'][jumps[i]]
                pot_good[i] = PCGA['Ewe/V'][jumps[i]]
            plt.plot(x,y,label="Linear fit on data before jump")
            plt.xlabel('<I>/mA')
            plt.ylabel('Ewe/V')
            plt.grid()
            plt.legend()
            plt.savefig('./results/PCGA_'+fnam+'.png')
            plt.show()
            PCGA_arr = {"col1": cur_good,"col2": pot_good}
            #print(PCGA_arr)
            PCGA_df = pd.DataFrame(PCGA_arr)
            plt.plot(PCGA['Ewe/V'])
            plt.xlabel("Time steps")
            plt.ylabel('Applied Potential [V]')
            plt.title('Polarization experiment: potential change over time')
            plt.show()
            #print(PCGA_df)
    return(PCGA_tot, PCGA_df)
